In [1]:
# ============================================================
# IMPORTS
# ============================================================
import copy
import json
import math
import random
import warnings
from pathlib import Path
import gc

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from sklearn.metrics import mean_absolute_error, mean_squared_error, median_absolute_error, r2_score
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier

warnings.filterwarnings("ignore")

In [2]:
# ============================================================
# REPRODUCIBILITY
# ============================================================
SEED = 42

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(SEED)

SKIP_COMPLETED = False

In [3]:
# ============================================================
# PATHS AND GLOBAL CONFIG
# ============================================================

PROJECT_DIR = Path(
    r"C:\Users\ANARAMO\Documents\Tese\Tese\Tese\HybridMobility"
)

DATA_DIR = PROJECT_DIR / "processed_data_final"

OUTPUT_DIR = DATA_DIR / "transformer_weighted_fusion_runs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("DEVICE:", DEVICE)

# -------------------------
# REGIME THRESHOLDS
# -------------------------
BRAKE_THRESHOLD = -0.1
ACCEL_THRESHOLD = 0.2

THRESHOLD_TAG = f"brake{abs(BRAKE_THRESHOLD):.1f}_accel{ACCEL_THRESHOLD:.1f}".replace(".", "p")

print("Using regime definition:")
print(f"  braking      : a_true < {BRAKE_THRESHOLD}")
print(f"  accelerating : a_true > {ACCEL_THRESHOLD}")
print("  steady       : otherwise")

# -------------------------
# MODEL / TRAINING CONFIG
# -------------------------

FPS = 25
HISTORY = 25

# 1, 13, 25, 50, 75, 125
HORIZON = 75

STRIDE = 1

HORIZON_SECONDS = HORIZON / FPS
HORIZON_TAG = f"H{HISTORY}_Hz{HORIZON}_S{STRIDE}"
SEQUENCE_TAG = HORIZON_TAG

BATCH_SIZE = 128
EPOCHS_BASE = 15
LR_BASE = 1e-3

D_MODEL = 64
NHEAD = 4
NUM_LAYERS = 2
DIM_FEEDFORWARD = 128
DROPOUT = 0.1

FUSION_VERSION_TAG = "currentStatePrior_v2"

print("History:", HISTORY)
print("Horizon:", HORIZON)
print("Horizon seconds:", HORIZON_SECONDS)
print("Stride:", STRIDE)
print("Sequence tag:", SEQUENCE_TAG)
# -------------------------
# EXPERIMENT SUITE CONFIG
# -------------------------

THRESHOLD_TAG = f"brake{abs(BRAKE_THRESHOLD):.1f}_accel{ACCEL_THRESHOLD:.1f}".replace(".", "p")

print("Threshold tag:", THRESHOLD_TAG)
print("Sequence tag:", SEQUENCE_TAG)

DEVICE: cuda
Using regime definition:
  braking      : a_true < -0.1
  accelerating : a_true > 0.2
  steady       : otherwise
History: 25
Horizon: 75
Horizon seconds: 3.0
Stride: 1
Sequence tag: H25_Hz75_S1
Threshold tag: brake0p1_accel0p2
Sequence tag: H25_Hz75_S1


In [4]:
# ============================================================
# LOAD PRE-SPLIT DATASETS
# ============================================================

required_files = [
    DATA_DIR / "episode_tracks_train.csv",
    DATA_DIR / "episode_tracks_val.csv",
    DATA_DIR / "episode_tracks_test.csv",
]

for p in required_files:
    assert p.exists(), f"Missing required file: {p}"

episode_tracks_train = pd.read_csv(DATA_DIR / "episode_tracks_train.csv")
episode_tracks_val   = pd.read_csv(DATA_DIR / "episode_tracks_val.csv")
episode_tracks_test  = pd.read_csv(DATA_DIR / "episode_tracks_test.csv")

print("train:", episode_tracks_train.shape)
print("val  :", episode_tracks_val.shape)
print("test :", episode_tracks_test.shape)

all_loaded = pd.concat(
    [episode_tracks_train, episode_tracks_val, episode_tracks_test],
    ignore_index=True
)

ego_split_check = all_loaded.groupby("unique_id")["split"].nunique()
episode_split_check = all_loaded.groupby("episode_id")["split"].nunique()

print("\nEgos in multiple splits:", (ego_split_check > 1).sum())
print("Episodes in multiple splits:", (episode_split_check > 1).sum())

assert (ego_split_check <= 1).all(), "Some egos appear in multiple splits."
assert (episode_split_check <= 1).all(), "Some episodes appear in multiple splits."

print("Split integrity checks passed.")

train: (11094352, 44)
val  : (2370107, 44)
test : (2368174, 44)

Egos in multiple splits: 0
Episodes in multiple splits: 0
Split integrity checks passed.


In [5]:
# ============================================================
# FEATURE PREPARATION
# Same base logic as IDM / Transformer notebooks
# ============================================================
def prepare_base_features(df):
    out = df.copy()

    # Direction-normalized speed features
    out["v_ego"] = out["xVelocity"].abs()
    out["v_leader"] = out["precedingXVelocity"].abs()

    # True signed acceleration target
    out["a_true"] = out["xAcceleration"]

    # Detect no-leader cases robustly
    no_leader = (
        out["precedingId"].isna()
        | (out["precedingId"] <= 0)
        | out["precedingXVelocity"].isna()
        | out["dhw"].isna()
    )

    # Relative speed and spacing
    out["delta_v"] = out["v_ego"] - out["v_leader"]
    out["gap"] = out["dhw"].clip(lower=0.1)

    # Free-road fallback for IDM-related calculations
    out.loc[no_leader, "gap"] = 1e6
    out.loc[no_leader, "delta_v"] = 0.0
    out.loc[no_leader, "v_leader"] = out.loc[no_leader, "v_ego"]

    out["no_leader"] = no_leader.astype(int)

    # Sort within episode for sequential modeling
    out = out.sort_values(["episode_id", "frame"]).reset_index(drop=True)

    return out


train_base = prepare_base_features(episode_tracks_train)
val_base   = prepare_base_features(episode_tracks_val)
test_base  = prepare_base_features(episode_tracks_test)

print(train_base[[
    "episode_id", "frame", "v_ego", "v_leader",
    "delta_v", "gap", "a_true", "no_leader"
]].head())

   episode_id  frame  v_ego  v_leader  delta_v    gap  a_true  no_leader
0           1  21549  41.19     43.10    -1.91  29.23   -0.39          0
1           1  21550  41.20     43.13    -1.93  29.46   -0.39          0
2           1  21551  41.22     43.15    -1.93  29.64   -0.40          0
3           1  21552  41.23     43.18    -1.95  29.78   -0.41          0
4           1  21553  41.25     43.20    -1.95  29.90   -0.42          0


In [6]:
# ============================================================
# LOAD HORIZON-SPECIFIC IDM PARAMETER SETS
# ============================================================

IDM_PARAM_TAG = "refinedIDM_v2"

IDM_CORE_RUN_DIR = (
    DATA_DIR
    / "idm_core_runs"
    / f"idm_core_{THRESHOLD_TAG}"
)

idm_config_path = (
    IDM_CORE_RUN_DIR
    / f"idm_core_config_{HORIZON_TAG}.json"
)

assert idm_config_path.exists(), (
    f"Missing horizon-specific IDM config:\n{idm_config_path}"
)

with open(idm_config_path, "r") as f:
    idm_config = json.load(f)

assert int(idm_config["HISTORY"]) == HISTORY
assert int(idm_config["HORIZON"]) == HORIZON
assert int(idm_config["STRIDE"]) == STRIDE

best_global_params = idm_config["best_global_params"]
best_params_braking = idm_config["best_params_braking"]
best_params_accel = idm_config["best_params_accel"]
best_params_steady = idm_config["best_params_steady"]

print("Loaded IDM configuration:")
print(idm_config_path)

print("\nGlobal:", best_global_params)
print("Braking:", best_params_braking)
print("Acceleration:", best_params_accel)
print("Steady:", best_params_steady)

Loaded IDM configuration:
C:\Users\ANARAMO\Documents\Tese\Tese\Tese\HybridMobility\processed_data_final\idm_core_runs\idm_core_brake0p1_accel0p2\idm_core_config_H25_Hz75_S1.json

Global: {'v0': 40.0, 'T': 0.5, 's0': 0.5, 'a': 0.4, 'b': 5.0, 'delta': 4.0}
Braking: {'v0': 1.0, 'T': 0.3, 's0': 5.0, 'a': 0.05, 'b': 26.0, 'delta': 0.5}
Acceleration: {'v0': 115.0, 'T': 0.1, 's0': 0.05, 'a': 0.4, 'b': 22.0, 'delta': 14.0}
Steady: {'v0': 85.0, 'T': 0.3, 's0': 0.1, 'a': 0.1, 'b': 8.0, 'delta': 2.0}


In [7]:
# ============================================================
# IDM ACCELERATION
# ============================================================
def idm_acceleration(v, delta_v, s, params):
    v0 = max(float(params["v0"]), 1e-6)
    T = float(params["T"])
    s0 = float(params["s0"])
    a = max(float(params["a"]), 1e-6)
    b = max(float(params["b"]), 1e-6)
    delta = float(params["delta"])

    v = np.maximum(v, 0.0)
    s = np.maximum(s, 0.1)

    s_star = s0 + np.maximum(0.0, v * T + (v * delta_v) / (2.0 * np.sqrt(a * b)))
    acc = a * (1.0 - (v / v0) ** delta - (s_star / s) ** 2)

    # clip like in your IDM implementation
    acc = np.clip(acc, -8.0, 4.0)
    return acc

In [8]:
# ============================================================
# BUILD FUTURE-REGIME SELECTOR DATA
#
# Features are observed at frame t.
# Regime label comes from acceleration at t + HORIZON.
# Rows match the Transformer sequence endpoints.
# ============================================================

selector_cols = ["v_ego", "v_leader", "delta_v", "gap"]


def make_future_selector_dataset(
    df,
    history=HISTORY,
    horizon=HORIZON,
    stride=STRIDE,
):
    parts = []

    for episode_id, g in df.groupby("episode_id", sort=False):
        g = g.sort_values("frame").reset_index(drop=True)

        frame_diff = g["frame"].diff().dropna()

        if not (frame_diff == 1).all():
            continue

        if len(g) < history + horizon:
            continue

        current_idx = np.arange(
            history - 1,
            len(g) - horizon,
            stride,
            dtype=int,
        )

        target_idx = current_idx + horizon

        current = g.iloc[current_idx].copy().reset_index(drop=True)
        future = g.iloc[target_idx].copy().reset_index(drop=True)

        current["input_frame"] = current["frame"].to_numpy()
        current["target_frame"] = future["frame"].to_numpy()
        current["a_true_future"] = future["a_true"].to_numpy()

        current["future_regime"] = np.select(
            [
                current["a_true_future"] < BRAKE_THRESHOLD,
                current["a_true_future"] > ACCEL_THRESHOLD,
            ],
            ["braking", "accelerating"],
            default="steady",
        )

        parts.append(current)

    if not parts:
        return pd.DataFrame()

    out = pd.concat(parts, ignore_index=True)

    frame_shift = out["target_frame"] - out["input_frame"]

    assert (frame_shift == horizon).all(), (
        f"Incorrect selector horizon: expected {horizon}, "
        f"found min={frame_shift.min()}, max={frame_shift.max()}"
    )

    return out


reg_train = make_future_selector_dataset(train_base)
reg_val = make_future_selector_dataset(val_base)
reg_test = make_future_selector_dataset(test_base)

X_reg_train = reg_train[selector_cols]
y_reg_train = reg_train["future_regime"]

selector_tree = DecisionTreeClassifier(
    max_depth=6,
    min_samples_leaf=500,
    random_state=SEED,
)

selector_tree.fit(X_reg_train, y_reg_train)

print("Future-regime tree selector trained.")
print("\nTrain rows:", len(reg_train))
print("Validation rows:", len(reg_val))
print("Test rows:", len(reg_test))

print("\nTest future-regime counts:")
print(reg_test["future_regime"].value_counts())

print("\nFrame-shift check:")
print(
    (
        reg_test["target_frame"]
        - reg_test["input_frame"]
    ).describe()
)

Future-regime tree selector trained.

Train rows: 7642222
Validation rows: 1630577
Test rows: 1628545

Test future-regime counts:
future_regime
steady          848187
accelerating    454352
braking         326006
Name: count, dtype: int64

Frame-shift check:
count    1628545.0
mean          75.0
std            0.0
min           75.0
25%           75.0
50%           75.0
75%           75.0
max           75.0
dtype: float64


In [9]:
# ============================================================
# BUILD IDM PRIOR
# Memory-safe version: keeps only columns needed downstream.
# ============================================================

def params_from_regime(reg):
    if reg == "braking":
        return best_params_braking
    elif reg == "accelerating":
        return best_params_accel
    else:
        return best_params_steady


def compute_piecewise_idm_prior(v, delta_v, gap, regimes):
    """
    Vectorized piecewise IDM prior.
    Avoids row-by-row loops.
    """

    v = np.asarray(v, dtype=np.float32)
    delta_v = np.asarray(delta_v, dtype=np.float32)
    gap = np.asarray(gap, dtype=np.float32)
    regimes = np.asarray(regimes)

    a_prior = np.empty(len(v), dtype=np.float32)

    for reg_name, params in [
        ("braking", best_params_braking),
        ("accelerating", best_params_accel),
        ("steady", best_params_steady),
    ]:
        mask = regimes == reg_name

        if mask.any():
            a_prior[mask] = idm_acceleration(
                v[mask],
                delta_v[mask],
                gap[mask],
                params
            )

    return np.clip(a_prior, -8, 4).astype(np.float32)


def build_prior_dataframe(df, prior_mode, selector=None):
    """
    Compute IDM priors from the state available at frame t.

    global:
        global IDM evaluated at t

    piecewise_tree:
        future-regime selector uses features at t and selects the
        regime-specific IDM evaluated at t

    oracle_piecewise:
        true future regime at t + HORIZON selects the IDM candidate,
        but IDM acceleration is still evaluated using the state at t
    """

    needed_cols = [
        "episode_id",
        "frame",
        "v_ego",
        "v_leader",
        "delta_v",
        "gap",
        "a_true",
    ]

    optional_cols = [
        c for c in ["unique_id", "split"]
        if c in df.columns
    ]

    out = df[needed_cols + optional_cols].copy()
    out = out.sort_values(["episode_id", "frame"]).reset_index(drop=True)

    float_cols = [
        "v_ego",
        "v_leader",
        "delta_v",
        "gap",
        "a_true",
    ]

    for col in float_cols:
        out[col] = out[col].astype(np.float32)

    # All IDM candidates are computed from the CURRENT state.
    out["a_idm_global"] = idm_acceleration(
        out["v_ego"].to_numpy(np.float32),
        out["delta_v"].to_numpy(np.float32),
        out["gap"].to_numpy(np.float32),
        best_global_params,
    ).astype(np.float32)

    out["a_idm_braking"] = idm_acceleration(
        out["v_ego"].to_numpy(np.float32),
        out["delta_v"].to_numpy(np.float32),
        out["gap"].to_numpy(np.float32),
        best_params_braking,
    ).astype(np.float32)

    out["a_idm_accel"] = idm_acceleration(
        out["v_ego"].to_numpy(np.float32),
        out["delta_v"].to_numpy(np.float32),
        out["gap"].to_numpy(np.float32),
        best_params_accel,
    ).astype(np.float32)

    out["a_idm_steady"] = idm_acceleration(
        out["v_ego"].to_numpy(np.float32),
        out["delta_v"].to_numpy(np.float32),
        out["gap"].to_numpy(np.float32),
        best_params_steady,
    ).astype(np.float32)

    if prior_mode == "global":
        out["a_idm_prior"] = out["a_idm_global"]

    elif prior_mode == "piecewise_tree":
        if selector is None:
            raise ValueError(
                "selector must be provided for piecewise_tree"
            )

        regime_pred = selector.predict(
            out[selector_cols].astype(np.float32)
        )

        out["regime_pred"] = regime_pred

        out["a_idm_prior"] = compute_piecewise_idm_prior(
            v=out["v_ego"].to_numpy(np.float32),
            delta_v=out["delta_v"].to_numpy(np.float32),
            gap=out["gap"].to_numpy(np.float32),
            regimes=regime_pred,
        )

    elif prior_mode == "oracle_piecewise":
        # Future acceleration aligned to every current row.
        out["a_true_future"] = (
            out.groupby("episode_id", sort=False)["a_true"]
            .shift(-HORIZON)
        )

        future_regime = np.select(
            [
                out["a_true_future"] < BRAKE_THRESHOLD,
                out["a_true_future"] > ACCEL_THRESHOLD,
            ],
            ["braking", "accelerating"],
            default="steady",
        )

        # Final HORIZON rows have no future target.
        future_regime = pd.Series(
            future_regime,
            index=out.index,
            dtype="object",
        )

        future_regime.loc[
            out["a_true_future"].isna()
        ] = np.nan

        out["regime"] = future_regime

        out["a_idm_prior"] = np.nan

        brake_mask = out["regime"] == "braking"
        accel_mask = out["regime"] == "accelerating"
        steady_mask = out["regime"] == "steady"

        out.loc[brake_mask, "a_idm_prior"] = out.loc[
            brake_mask, "a_idm_braking"
        ]

        out.loc[accel_mask, "a_idm_prior"] = out.loc[
            accel_mask, "a_idm_accel"
        ]

        out.loc[steady_mask, "a_idm_prior"] = out.loc[
            steady_mask, "a_idm_steady"
        ]

        # These ending rows cannot be sequence endpoints anyway.
        out["a_idm_prior"] = out["a_idm_prior"].fillna(
            out["a_idm_global"]
        )

    else:
        raise ValueError(f"Unknown prior_mode: {prior_mode}")

    out["a_idm_prior"] = (
        out["a_idm_prior"]
        .clip(-8, 4)
        .astype(np.float32)
    )

    return out
 

In [10]:
# ============================================================
# BUILD TEMPORARY GLOBAL PRIOR FOR BASE TRANSFORMER SEQUENCES
# The base Transformer does not use IDM as input, but make_sequences
# expects a_idm_prior in the metadata. Global IDM is used only to
# build aligned metadata for the base model.
# ============================================================

base_train_prior = build_prior_dataframe(train_base, prior_mode="global", selector=None)
base_val_prior   = build_prior_dataframe(val_base, prior_mode="global", selector=None)
base_test_prior  = build_prior_dataframe(test_base, prior_mode="global", selector=None)

print("Temporary global prior created for base Transformer sequence construction.")
print("train:", base_train_prior.shape)
print("val  :", base_val_prior.shape)
print("test :", base_test_prior.shape)

Temporary global prior created for base Transformer sequence construction.
train: (11094352, 14)
val  : (2370107, 14)
test : (2368174, 14)


In [11]:
# ============================================================
# BUILD BASE TRANSFORMER SEQUENCES
#
# History window ends at frame t.
# Transformer target is acceleration at t + HORIZON.
# IDM prior stored in metadata comes from frame t.
# ============================================================

BASE_FEATURE_COLS = [
    "v_ego",
    "v_leader",
    "delta_v",
    "gap",
]


def make_sequences(
    df,
    history=HISTORY,
    horizon=HORIZON,
    stride=STRIDE,
):
    X_seq = []
    y_target = []
    meta_rows = []

    for episode_id, g in df.groupby("episode_id", sort=False):
        g = g.sort_values("frame").reset_index(drop=True)

        frame_diff = g["frame"].diff().dropna()

        if not (frame_diff == 1).all():
            continue

        if len(g) < history + horizon:
            continue

        values = g[BASE_FEATURE_COLS].to_numpy(dtype=np.float32)
        a_true = g["a_true"].to_numpy(dtype=np.float32)
        a_idm = g["a_idm_prior"].to_numpy(dtype=np.float32)

        for current_idx in range(
            history - 1,
            len(g) - horizon,
            stride,
        ):
            history_start = current_idx - history + 1
            target_idx = current_idx + horizon

            X_seq.append(
                values[history_start : current_idx + 1]
            )

            y_target.append(
                a_true[target_idx]
            )

            row = {
                "episode_id": episode_id,

                # Current observation and future target frames
                "input_frame": g.loc[current_idx, "frame"],
                "target_frame": g.loc[target_idx, "frame"],
                "frame": g.loc[target_idx, "frame"],

                # Future acceleration target
                "a_true": a_true[target_idx],

                # Critical: current-state IDM prior
                "a_idm_prior": a_idm[current_idx],

                # Current-state variables used by the IDM
                "v_ego_input": g.loc[current_idx, "v_ego"],
                "v_leader_input": g.loc[current_idx, "v_leader"],
                "delta_v_input": g.loc[current_idx, "delta_v"],
                "gap_input": g.loc[current_idx, "gap"],

                # Future target-frame values for diagnostics only
                "v_ego": g.loc[target_idx, "v_ego"],
                "v_leader": g.loc[target_idx, "v_leader"],
                "delta_v": g.loc[target_idx, "delta_v"],
                "gap": g.loc[target_idx, "gap"],

                "history_frames": history,
                "horizon_frames": horizon,
                "horizon_seconds": horizon / FPS,
                "stride": stride,
            }

            if "unique_id" in g.columns:
                row["unique_id"] = g.loc[target_idx, "unique_id"]

            if "split" in g.columns:
                row["split"] = g.loc[target_idx, "split"]

            # True future regime
            future_a = a_true[target_idx]

            if future_a < BRAKE_THRESHOLD:
                row["regime"] = "braking"
            elif future_a > ACCEL_THRESHOLD:
                row["regime"] = "accelerating"
            else:
                row["regime"] = "steady"

            # Selector output belongs to the current frame
            if "regime_pred" in g.columns:
                row["regime_pred"] = g.loc[current_idx, "regime_pred"]

            meta_rows.append(row)

    X_seq = np.asarray(X_seq, dtype=np.float32)
    y_target = np.asarray(y_target, dtype=np.float32).reshape(-1, 1)
    meta_df = pd.DataFrame(meta_rows)

    assert len(X_seq) == len(y_target) == len(meta_df), (
        f"Length mismatch: X={len(X_seq)}, "
        f"y={len(y_target)}, meta={len(meta_df)}"
    )

    if len(meta_df) > 0:
        frame_shift = (
            meta_df["target_frame"]
            - meta_df["input_frame"]
        )

        assert (frame_shift == horizon).all(), (
            f"Wrong sequence horizon. Expected {horizon}; "
            f"observed min={frame_shift.min()}, "
            f"max={frame_shift.max()}."
        )

    return X_seq, y_target, meta_df


X_train, y_train, meta_train = make_sequences(
    base_train_prior,
    history=HISTORY,
    horizon=HORIZON,
    stride=STRIDE,
)

X_val, y_val, meta_val = make_sequences(
    base_val_prior,
    history=HISTORY,
    horizon=HORIZON,
    stride=STRIDE,
)

X_test, y_test, meta_test = make_sequences(
    base_test_prior,
    history=HISTORY,
    horizon=HORIZON,
    stride=STRIDE,
)

print("X_train:", X_train.shape)
print("y_train:", y_train.shape)
print("meta_train:", meta_train.shape)

print("\nX_val:", X_val.shape)
print("y_val:", y_val.shape)
print("meta_val:", meta_val.shape)

print("\nX_test:", X_test.shape)
print("y_test:", y_test.shape)
print("meta_test:", meta_test.shape)

print("\nSequence horizon check:")
print(
    (
        meta_test["target_frame"]
        - meta_test["input_frame"]
    ).describe()
)

X_train: (7642222, 25, 4)
y_train: (7642222, 1)
meta_train: (7642222, 21)

X_val: (1630577, 25, 4)
y_val: (1630577, 1)
meta_val: (1630577, 21)

X_test: (1628545, 25, 4)
y_test: (1628545, 1)
meta_test: (1628545, 21)

Sequence horizon check:
count    1628545.0
mean          75.0
std            0.0
min           75.0
25%           75.0
50%           75.0
75%           75.0
max           75.0
dtype: float64


In [12]:
# ============================================================
# BUILD CURRENT-STATE IDM PRIOR METADATA
#
# Transformer history ends at frame t.
# IDM prior is computed at frame t.
# Prediction target is acceleration at t + HORIZON.
# ============================================================

def make_prior_meta_only(
    df,
    history=HISTORY,
    horizon=HORIZON,
    stride=STRIDE,
):
    meta_rows = []

    for episode_id, g in df.groupby("episode_id", sort=False):
        g = g.sort_values("frame").reset_index(drop=True)

        frame_diff = g["frame"].diff().dropna()

        if not (frame_diff == 1).all():
            continue

        if len(g) < history + horizon:
            continue

        a_true = g["a_true"].to_numpy(dtype=np.float32)
        a_idm = g["a_idm_prior"].to_numpy(dtype=np.float32)

        for current_idx in range(
            history - 1,
            len(g) - horizon,
            stride,
        ):
            target_idx = current_idx + horizon

            row = {
                "episode_id": episode_id,

                "input_frame": g.loc[current_idx, "frame"],
                "target_frame": g.loc[target_idx, "frame"],
                "frame": g.loc[target_idx, "frame"],

                "a_true": a_true[target_idx],

                # Critical: current-state IDM prior
                "a_idm_prior": a_idm[current_idx],

                # Current-state values used by IDM
                "v_ego_input": g.loc[current_idx, "v_ego"],
                "v_leader_input": g.loc[current_idx, "v_leader"],
                "delta_v_input": g.loc[current_idx, "delta_v"],
                "gap_input": g.loc[current_idx, "gap"],

                # Future target-state diagnostics
                "v_ego": g.loc[target_idx, "v_ego"],
                "v_leader": g.loc[target_idx, "v_leader"],
                "delta_v": g.loc[target_idx, "delta_v"],
                "gap": g.loc[target_idx, "gap"],

                "history_frames": history,
                "horizon_frames": horizon,
                "horizon_seconds": horizon / FPS,
                "stride": stride,
            }

            if "unique_id" in g.columns:
                row["unique_id"] = g.loc[target_idx, "unique_id"]

            if "split" in g.columns:
                row["split"] = g.loc[target_idx, "split"]

            future_a = a_true[target_idx]

            if future_a < BRAKE_THRESHOLD:
                row["regime"] = "braking"
            elif future_a > ACCEL_THRESHOLD:
                row["regime"] = "accelerating"
            else:
                row["regime"] = "steady"

            # Selector prediction belongs to current state
            if "regime_pred" in g.columns:
                row["regime_pred"] = g.loc[current_idx, "regime_pred"]

            meta_rows.append(row)

    meta_df = pd.DataFrame(meta_rows)

    if len(meta_df) > 0:
        frame_shift = (
            meta_df["target_frame"]
            - meta_df["input_frame"]
        )

        assert (frame_shift == horizon).all(), (
            f"Wrong prior alignment. Expected {horizon}; "
            f"observed min={frame_shift.min()}, "
            f"max={frame_shift.max()}."
        )

    return meta_df

In [13]:
# ============================================================
# SCALE INPUTS AND TARGETS
# ============================================================
X_scaler = StandardScaler()
y_scaler = StandardScaler()

# fit X scaler on flattened train sequences
X_train_2d = X_train.reshape(-1, X_train.shape[-1])
X_scaler.fit(X_train_2d)

def scale_X(X, scaler):
    n, h, d = X.shape
    X2 = X.reshape(-1, d)
    X2s = scaler.transform(X2)
    return X2s.reshape(n, h, d).astype(np.float32)

X_train_s = scale_X(X_train, X_scaler)
X_val_s   = scale_X(X_val, X_scaler)
X_test_s  = scale_X(X_test, X_scaler)

y_scaler.fit(y_train)
y_train_s = y_scaler.transform(y_train).astype(np.float32)
y_val_s   = y_scaler.transform(y_val).astype(np.float32)
y_test_s  = y_scaler.transform(y_test).astype(np.float32)

print("Scaled.")

Scaled.


In [14]:
# ============================================================
# BASE TRANSFORMER DATASET
# ============================================================
class BaseSequenceDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

train_ds_base = BaseSequenceDataset(X_train_s, y_train_s)
val_ds_base   = BaseSequenceDataset(X_val_s, y_val_s)
test_ds_base  = BaseSequenceDataset(X_test_s, y_test_s)

train_loader_base = DataLoader(train_ds_base, batch_size=BATCH_SIZE, shuffle=True)
val_loader_base   = DataLoader(val_ds_base, batch_size=BATCH_SIZE, shuffle=False)
test_loader_base  = DataLoader(test_ds_base, batch_size=BATCH_SIZE, shuffle=False)

In [15]:
# ============================================================
# TRANSFORMER BASE MODEL
# ============================================================
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float32).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))

        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)

        pe = pe.unsqueeze(0)  # (1, max_len, d_model)
        self.register_buffer("pe", pe)

    def forward(self, x):
        return x + self.pe[:, :x.size(1), :]

class BaseTransformerRegressor(nn.Module):
    def __init__(
        self,
        input_dim,
        d_model=64,
        nhead=4,
        num_layers=2,
        dim_feedforward=128,
        dropout=0.1
    ):
        super().__init__()

        self.input_proj = nn.Linear(input_dim, d_model)
        self.pos_encoder = PositionalEncoding(d_model)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            batch_first=True
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)

        self.head = nn.Sequential(
            nn.Linear(d_model, d_model),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(d_model, 1)
        )

    def forward(self, x, return_hidden=False):
        h = self.input_proj(x)
        h = self.pos_encoder(h)
        h = self.encoder(h)

        # last token representation
        h_last = h[:, -1, :]
        y = self.head(h_last)

        if return_hidden:
            return y, h_last
        return y

In [16]:
# ============================================================
# TRAIN / EVAL UTILS
# ============================================================
def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()
    losses = []

    for Xb, yb in loader:
        Xb = Xb.to(device)
        yb = yb.to(device)

        optimizer.zero_grad()
        pred = model(Xb)
        loss = criterion(pred, yb)
        loss.backward()
        optimizer.step()

        losses.append(loss.item())

    return float(np.mean(losses))

@torch.no_grad()
def evaluate_model(model, loader, criterion, device):
    model.eval()
    losses = []
    preds = []
    targets = []

    for Xb, yb in loader:
        Xb = Xb.to(device)
        yb = yb.to(device)

        pred = model(Xb)
        loss = criterion(pred, yb)

        losses.append(loss.item())
        preds.append(pred.cpu().numpy())
        targets.append(yb.cpu().numpy())

    preds = np.vstack(preds)
    targets = np.vstack(targets)

    return float(np.mean(losses)), preds, targets

In [17]:
# ============================================================
# TRAIN BASE TRANSFORMER
# Saves best checkpoint immediately and last checkpoint every epoch.
# ============================================================

base_ckpt_path = OUTPUT_DIR / f"base_transformer_{SEQUENCE_TAG}_{THRESHOLD_TAG}.pt"
base_last_ckpt_path = OUTPUT_DIR / f"base_transformer_{SEQUENCE_TAG}_{THRESHOLD_TAG}_last.pt"
base_log_path = OUTPUT_DIR / f"base_transformer_{SEQUENCE_TAG}_{THRESHOLD_TAG}_training_log.csv"

base_model = BaseTransformerRegressor(
    input_dim=X_train_s.shape[-1],
    d_model=D_MODEL,
    nhead=NHEAD,
    num_layers=NUM_LAYERS,
    dim_feedforward=DIM_FEEDFORWARD,
    dropout=DROPOUT
).to(DEVICE)

criterion = nn.MSELoss()
optimizer = torch.optim.Adam(base_model.parameters(), lr=LR_BASE)

best_val_loss = float("inf")
best_state = None
best_epoch = -1
train_losses = []
val_losses = []
log_rows = []

if base_ckpt_path.exists():
    print("Loading existing base transformer checkpoint...")
    checkpoint = torch.load(base_ckpt_path, map_location=DEVICE)

    if isinstance(checkpoint, dict) and "model_state_dict" in checkpoint:
        base_model.load_state_dict(checkpoint["model_state_dict"])
        best_val_loss = checkpoint.get("best_val_loss", np.nan)
        best_epoch = checkpoint.get("best_epoch", -1)
        print("Loaded best epoch:", best_epoch)
        print("Loaded best val loss:", best_val_loss)
    else:
        base_model.load_state_dict(checkpoint)

else:
    print("Training base transformer...")

    for epoch in range(1, EPOCHS_BASE + 1):
        train_loss = train_one_epoch(
            base_model,
            train_loader_base,
            optimizer,
            criterion,
            DEVICE
        )

        val_loss, _, _ = evaluate_model(
            base_model,
            val_loader_base,
            criterion,
            DEVICE
        )

        train_losses.append(train_loss)
        val_losses.append(val_loss)

        improved = val_loss < best_val_loss

        if improved:
            best_val_loss = val_loss
            best_state = copy.deepcopy(base_model.state_dict())
            best_epoch = epoch

            torch.save(
                {
                    "model_state_dict": best_state,
                    "optimizer_state_dict": optimizer.state_dict(),
                    "best_val_loss": best_val_loss,
                    "best_epoch": best_epoch,
                    "train_losses": train_losses,
                    "val_losses": val_losses,
                    "HISTORY": HISTORY,
                    "HORIZON": HORIZON,
                    "STRIDE": STRIDE,
                    "SEQUENCE_TAG": SEQUENCE_TAG,
                    "THRESHOLD_TAG": THRESHOLD_TAG,
                    "model_config": {
                        "input_dim": X_train_s.shape[-1],
                        "d_model": D_MODEL,
                        "nhead": NHEAD,
                        "num_layers": NUM_LAYERS,
                        "dim_feedforward": DIM_FEEDFORWARD,
                        "dropout": DROPOUT,
                    }
                },
                base_ckpt_path
            )

        torch.save(
            {
                "model_state_dict": base_model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "best_val_loss": best_val_loss,
                "best_epoch": best_epoch,
                "completed_epoch": epoch,
                "train_losses": train_losses,
                "val_losses": val_losses,
                "HISTORY": HISTORY,
                "HORIZON": HORIZON,
                "STRIDE": STRIDE,
                "SEQUENCE_TAG": SEQUENCE_TAG,
                "THRESHOLD_TAG": THRESHOLD_TAG,
                "model_config": {
                    "input_dim": X_train_s.shape[-1],
                    "d_model": D_MODEL,
                    "nhead": NHEAD,
                    "num_layers": NUM_LAYERS,
                    "dim_feedforward": DIM_FEEDFORWARD,
                    "dropout": DROPOUT,
                }
            },
            base_last_ckpt_path
        )

        log_rows.append({
            "epoch": epoch,
            "train_loss": train_loss,
            "val_loss": val_loss,
            "best_val_loss": best_val_loss,
            "best_epoch": best_epoch,
            "improved": improved,
        })

        pd.DataFrame(log_rows).to_csv(base_log_path, index=False)

        status = " | improved + saved best" if improved else ""

        print(
            f"[Base][Epoch {epoch:02d}] "
            f"train_loss={train_loss:.6f} "
            f"val_loss={val_loss:.6f}{status}"
        )

    # Load best state after training
    best_checkpoint = torch.load(base_ckpt_path, map_location=DEVICE)
    base_model.load_state_dict(best_checkpoint["model_state_dict"])

print("Base transformer ready.")
print("Best checkpoint:", base_ckpt_path)
print("Last checkpoint:", base_last_ckpt_path)
print("Training log:", base_log_path)

Training base transformer...
[Base][Epoch 01] train_loss=0.798526 val_loss=0.757116 | improved + saved best
[Base][Epoch 02] train_loss=0.778476 val_loss=0.739847 | improved + saved best
[Base][Epoch 03] train_loss=0.768366 val_loss=0.740589
[Base][Epoch 04] train_loss=0.761575 val_loss=0.737315 | improved + saved best
[Base][Epoch 05] train_loss=0.756562 val_loss=0.749419
[Base][Epoch 06] train_loss=0.752257 val_loss=0.748923
[Base][Epoch 07] train_loss=0.748876 val_loss=0.772211
[Base][Epoch 08] train_loss=0.744321 val_loss=0.754331
[Base][Epoch 09] train_loss=0.739473 val_loss=0.760317
[Base][Epoch 10] train_loss=0.737779 val_loss=0.749420
[Base][Epoch 11] train_loss=0.736114 val_loss=0.760759
[Base][Epoch 12] train_loss=0.733393 val_loss=0.743381
[Base][Epoch 13] train_loss=0.730759 val_loss=0.766705
[Base][Epoch 14] train_loss=0.728051 val_loss=0.786862
[Base][Epoch 15] train_loss=0.725011 val_loss=0.759110
Base transformer ready.
Best checkpoint: C:\Users\ANARAMO\Documents\Tese\T

In [18]:
# ============================================================
# GET BASE TRANSFORMER PREDICTIONS
# ============================================================
def predict_base(model, X_scaled, batch_size=512):
    ds = BaseSequenceDataset(X_scaled, np.zeros((len(X_scaled), 1), dtype=np.float32))
    loader = DataLoader(ds, batch_size=batch_size, shuffle=False)

    model.eval()
    preds = []
    hidden = []

    with torch.no_grad():
        for Xb, _ in loader:
            Xb = Xb.to(DEVICE)
            pred_b, hid_b = model(Xb, return_hidden=True)
            preds.append(pred_b.cpu().numpy())
            hidden.append(hid_b.cpu().numpy())

    preds = np.vstack(preds)
    hidden = np.vstack(hidden)
    return preds, hidden

base_train_pred_s, base_train_hidden = predict_base(base_model, X_train_s)
base_val_pred_s,   base_val_hidden   = predict_base(base_model, X_val_s)
base_test_pred_s,  base_test_hidden  = predict_base(base_model, X_test_s)

base_train_pred = y_scaler.inverse_transform(base_train_pred_s)
base_val_pred   = y_scaler.inverse_transform(base_val_pred_s)
base_test_pred  = y_scaler.inverse_transform(base_test_pred_s)

# Original-scale targets for weighted fusion evaluation
y_train_orig = y_scaler.inverse_transform(y_train_s)
y_val_orig   = y_scaler.inverse_transform(y_val_s)
y_test_orig  = y_scaler.inverse_transform(y_test_s)

print("Base predictions ready.")
print("base_train_pred:", base_train_pred.shape, "y_train_orig:", y_train_orig.shape)
print("base_val_pred  :", base_val_pred.shape, "y_val_orig  :", y_val_orig.shape)
print("base_test_pred :", base_test_pred.shape, "y_test_orig :", y_test_orig.shape)

Base predictions ready.
base_train_pred: (7642222, 1) y_train_orig: (7642222, 1)
base_val_pred  : (1630577, 1) y_val_orig  : (1630577, 1)
base_test_pred : (1628545, 1) y_test_orig : (1628545, 1)


In [19]:
# ============================================================
# FREE BASE SEQUENCE ARRAYS AFTER BASE TRANSFORMER PREDICTION
# ============================================================

# At this point, weighted fusion only needs:
# - base_train_pred, base_val_pred, base_test_pred
# - y_train_orig, y_val_orig, y_test_orig
# - meta_train, meta_val, meta_test
#
# The full sequence arrays are no longer needed.

for name in [
    "X_train", "X_val", "X_test",
    "X_train_s", "X_val_s", "X_test_s"
]:
    if name in globals():
        del globals()[name]

gc.collect()

print("Freed base Transformer sequence arrays.")

Freed base Transformer sequence arrays.


In [20]:
# ============================================================
# FINAL WEIGHTED FUSION EXPERIMENT SUITE
# Runs all final weighted-fusion experiments in one execution.
# Saves each experiment immediately, so completed runs are preserved.
# ============================================================

def compute_metrics(y_true, y_pred):
    y_true = np.asarray(y_true).reshape(-1)
    y_pred = np.asarray(y_pred).reshape(-1)

    corr = np.nan
    if len(y_true) > 1 and np.std(y_true) > 0 and np.std(y_pred) > 0:
        corr = float(np.corrcoef(y_true, y_pred)[0, 1])

    return {
        "MAE": mean_absolute_error(y_true, y_pred),
        "RMSE": np.sqrt(mean_squared_error(y_true, y_pred)),
        "MedAE": median_absolute_error(y_true, y_pred),
        "R2": r2_score(y_true, y_pred),
        "Bias": float(np.mean(y_pred - y_true)),
        "Corr": corr,
    }


def weighted_fusion(transformer_pred, idm_pred, alpha):
    """
    alpha = Transformer weight
    1-alpha = IDM weight
    """
    return alpha * transformer_pred + (1.0 - alpha) * idm_pred


def find_best_alpha(y_true_val, transformer_val, idm_val, metric="MAE"):
    alphas = np.linspace(0.0, 1.0, 101)

    rows = []

    for alpha in alphas:
        pred = weighted_fusion(transformer_val, idm_val, alpha)
        m = compute_metrics(y_true_val, pred)

        rows.append({
            "alpha": alpha,
            **m
        })

    alpha_search_df = pd.DataFrame(rows)

    if metric == "MAE":
        best_row = alpha_search_df.sort_values("MAE").iloc[0]
    elif metric == "RMSE":
        best_row = alpha_search_df.sort_values("RMSE").iloc[0]
    else:
        raise ValueError("metric must be MAE or RMSE")

    return float(best_row["alpha"]), alpha_search_df


def add_regime_from_target(df, y_col="y_true"):
    out = df.copy()

    out["regime"] = np.where(
        out[y_col] < BRAKE_THRESHOLD,
        "braking",
        np.where(
            out[y_col] > ACCEL_THRESHOLD,
            "accelerating",
            "steady"
        )
    )

    return out


def regime_metrics_from_results(results_df):
    out = (
        results_df.groupby("regime")
        .apply(lambda g: pd.Series({
            "MAE": np.mean(np.abs(g["y_true"] - g["y_pred"])),
            "RMSE": np.sqrt(np.mean((g["y_true"] - g["y_pred"]) ** 2)),
            "Bias": np.mean(g["y_pred"] - g["y_true"]),
            "Count": len(g)
        }))
        .reset_index()
    )

    regime_order = ["accelerating", "braking", "steady"]

    out["regime"] = pd.Categorical(
        out["regime"],
        categories=regime_order,
        ordered=True
    )

    return out.sort_values("regime").reset_index(drop=True)


FINAL_PRIOR_EXPERIMENTS = [
    {
        "prior_mode": "global",
        "prior_label": "GlobalIDM",
        "model_label": "Weighted Fusion: Transformer + Global IDM",
        "selector": None,
    },
    {
        "prior_mode": "piecewise_tree",
        "prior_label": "TreeRegimeAwareIDM",
        "model_label": "Weighted Fusion: Transformer + Tree-based regime-aware IDM",
        "selector": selector_tree,
    },
    {
        "prior_mode": "oracle_piecewise",
        "prior_label": "IdealRegimeAwareIDM",
        "model_label": "Weighted Fusion: Transformer + ideal regime-aware IDM",
        "selector": None,
    },
]



suite_rows = []
suite_regime_rows = []
suite_alpha_rows = []

for exp in FINAL_PRIOR_EXPERIMENTS:
    prior_mode = exp["prior_mode"]
    prior_label = exp["prior_label"]
    model_label = exp["model_label"]
    selector = exp["selector"]

    experiment_name = (
        f"TransformerWeightedFusion_{prior_label}_"
        f"{SEQUENCE_TAG}_{THRESHOLD_TAG}_{IDM_PARAM_TAG}_"
        f"{FUSION_VERSION_TAG}"
    )
    run_dir = OUTPUT_DIR / experiment_name
    run_dir.mkdir(parents=True, exist_ok=True)

    summary_path = run_dir / "weighted_fusion_summary.csv"
    split_path = run_dir / "weighted_fusion_split_metrics.csv"
    regime_path = run_dir / "weighted_fusion_regime_metrics.csv"
    predictions_path = run_dir / "weighted_fusion_test_predictions.csv"
    alpha_path = run_dir / "weighted_fusion_alpha_search.csv"
    config_path = run_dir / "config.json"

    print("\n" + "=" * 80)
    print("Experiment:", experiment_name)
    print("Prior mode:", prior_mode)
    print("Run dir:", run_dir)
    print("=" * 80)

    if (
        SKIP_COMPLETED
        and summary_path.exists()
        and split_path.exists()
        and regime_path.exists()
        and predictions_path.exists()
        and alpha_path.exists()
    ):
        print("Completed outputs found. Loading existing results instead of rerunning.")

        existing_summary = pd.read_csv(summary_path)
        existing_regime = pd.read_csv(regime_path)
        existing_alpha = pd.read_csv(alpha_path)

        row = existing_summary.iloc[0].to_dict()
        row["ExperimentName"] = experiment_name
        row["PriorMode"] = prior_mode
        row["Model"] = model_label

        suite_rows.append(row)

        temp_regime = existing_regime.copy()
        temp_regime["ExperimentName"] = experiment_name
        temp_regime["PriorMode"] = prior_mode
        temp_regime["Model"] = model_label
        suite_regime_rows.append(temp_regime)

        temp_alpha = existing_alpha.copy()
        temp_alpha["ExperimentName"] = experiment_name
        temp_alpha["PriorMode"] = prior_mode
        suite_alpha_rows.append(temp_alpha)

        continue

    # --------------------------------------------------------
    # Build IDM prior for this experiment
    # --------------------------------------------------------
    train_prior = build_prior_dataframe(train_base, prior_mode=prior_mode, selector=selector)
    val_prior   = build_prior_dataframe(val_base, prior_mode=prior_mode, selector=selector)
    test_prior  = build_prior_dataframe(test_base, prior_mode=prior_mode, selector=selector)

    meta_train_p = make_prior_meta_only(train_prior, HISTORY, HORIZON, STRIDE)
    meta_val_p   = make_prior_meta_only(val_prior, HISTORY, HORIZON, STRIDE)
    meta_test_p  = make_prior_meta_only(test_prior, HISTORY, HORIZON, STRIDE)

    # Free full frame-level prior dataframes as soon as metadata is built
    del train_prior, val_prior, test_prior
    gc.collect()

    # Safety check: base sequences and prior sequences must align
    assert len(meta_train_p) == len(base_train_pred), "Train sequence length mismatch."
    assert len(meta_val_p) == len(base_val_pred), "Val sequence length mismatch."
    assert len(meta_test_p) == len(base_test_pred), "Test sequence length mismatch."

    # --------------------------------------------------------
    # Exact timestamp and target alignment checks
    # --------------------------------------------------------

    assert np.array_equal(
        meta_train_p["target_frame"].to_numpy(),
        meta_train["target_frame"].to_numpy(),
    ), "Train target-frame mismatch."

    assert np.array_equal(
        meta_val_p["target_frame"].to_numpy(),
        meta_val["target_frame"].to_numpy(),
    ), "Validation target-frame mismatch."

    assert np.array_equal(
        meta_test_p["target_frame"].to_numpy(),
        meta_test["target_frame"].to_numpy(),
    ), "Test target-frame mismatch."

    assert np.array_equal(
        meta_train_p["input_frame"].to_numpy(),
        meta_train["input_frame"].to_numpy(),
    ), "Train input-frame mismatch."

    assert np.array_equal(
        meta_val_p["input_frame"].to_numpy(),
        meta_val["input_frame"].to_numpy(),
    ), "Validation input-frame mismatch."

    assert np.array_equal(
        meta_test_p["input_frame"].to_numpy(),
        meta_test["input_frame"].to_numpy(),
    ), "Test input-frame mismatch."

    assert (
        meta_test_p["target_frame"]
        - meta_test_p["input_frame"]
    ).eq(HORIZON).all(), "Incorrect test horizon."

    assert np.allclose(
        meta_test_p["a_true"].to_numpy(),
        y_test_orig.reshape(-1),
        atol=1e-6,
    ), "Test target acceleration mismatch."

    print("Prior metadata aligned correctly with Transformer targets.")
        

    idm_train = meta_train_p[["a_idm_prior"]].values
    idm_val = meta_val_p[["a_idm_prior"]].values
    idm_test = meta_test_p[["a_idm_prior"]].values

    # --------------------------------------------------------
    # Select best global alpha on validation
    # --------------------------------------------------------
    best_alpha, alpha_search_df = find_best_alpha(
        y_true_val=y_val_orig,
        transformer_val=base_val_pred,
        idm_val=idm_val,
        metric="MAE"
    )

    print("Best alpha selected on validation:", best_alpha)
    print("Transformer weight:", best_alpha)
    print("IDM weight:", 1.0 - best_alpha)

    # --------------------------------------------------------
    # Final fusion predictions
    # --------------------------------------------------------
    fusion_train_pred = weighted_fusion(base_train_pred, idm_train, best_alpha)
    fusion_val_pred = weighted_fusion(base_val_pred, idm_val, best_alpha)
    fusion_test_pred = weighted_fusion(base_test_pred, idm_test, best_alpha)

    metrics_train = compute_metrics(y_train_orig, fusion_train_pred)
    metrics_val = compute_metrics(y_val_orig, fusion_val_pred)
    metrics_test = compute_metrics(y_test_orig, fusion_test_pred)

    fusion_split_metrics_df = pd.DataFrame([
        {"split": "train", **metrics_train},
        {"split": "val", **metrics_val},
        {"split": "test", **metrics_test},
    ])

    fusion_test_results = meta_test_p.copy()
    fusion_test_results["y_true"] = y_test_orig.reshape(-1)
    fusion_test_results["y_pred"] = fusion_test_pred.reshape(-1)
    fusion_test_results["a_transformer_base"] = base_test_pred.reshape(-1)
    fusion_test_results["a_idm_prior"] = idm_test.reshape(-1)
    fusion_test_results["alpha_transformer"] = best_alpha
    fusion_test_results["alpha_idm"] = 1.0 - best_alpha
    fusion_test_results["error"] = fusion_test_results["y_pred"] - fusion_test_results["y_true"]
    fusion_test_results["abs_error"] = fusion_test_results["error"].abs()
    fusion_test_results["Model"] = model_label
    fusion_test_results["PriorMode"] = prior_mode

    fusion_test_results = add_regime_from_target(fusion_test_results, y_col="y_true")

    fusion_regime_metrics_df = regime_metrics_from_results(fusion_test_results)

    fusion_summary_df = pd.DataFrame([
        {
            "ExperimentName": experiment_name,
            "Model": model_label,
            "PriorMode": prior_mode,
            "BrakeThreshold": BRAKE_THRESHOLD,
            "AccelThreshold": ACCEL_THRESHOLD,
            "alpha_transformer": best_alpha,
            "alpha_idm": 1.0 - best_alpha,
            **metrics_test
        }
    ])

    alpha_search_df["ExperimentName"] = experiment_name
    alpha_search_df["PriorMode"] = prior_mode

    # --------------------------------------------------------
    # Save immediately
    # --------------------------------------------------------
    fusion_summary_df.to_csv(summary_path, index=False)
    fusion_split_metrics_df.to_csv(split_path, index=False)
    fusion_regime_metrics_df.to_csv(regime_path, index=False)
    fusion_test_results.to_csv(predictions_path, index=False)
    alpha_search_df.to_csv(alpha_path, index=False)

    with open(config_path, "w") as f:
        json.dump({
            "EXPERIMENT_NAME": experiment_name,
            "MODEL_LABEL": model_label,
            "PRIOR_MODE": prior_mode,
            "IDM_PARAM_TAG": IDM_PARAM_TAG,
            "SEQUENCE_TAG": SEQUENCE_TAG,
            "BRAKE_THRESHOLD": BRAKE_THRESHOLD,
            "ACCEL_THRESHOLD": ACCEL_THRESHOLD,
            "HISTORY": HISTORY,
            "HORIZON": HORIZON,
            "FPS": FPS,
            "HORIZON_SECONDS": HORIZON_SECONDS,
            "HORIZON_TAG": HORIZON_TAG,
            "IDM_CONFIG_PATH": str(idm_config_path),
            "FUSION_VERSION_TAG": FUSION_VERSION_TAG,
            "STRIDE": STRIDE,
            "BATCH_SIZE": BATCH_SIZE,
            "EPOCHS_BASE": EPOCHS_BASE,
            "LR_BASE": LR_BASE,
            "D_MODEL": D_MODEL,
            "NHEAD": NHEAD,
            "NUM_LAYERS": NUM_LAYERS,
            "DIM_FEEDFORWARD": DIM_FEEDFORWARD,
            "DROPOUT": DROPOUT,
            "BASE_FEATURE_COLS": BASE_FEATURE_COLS,
            "best_alpha_transformer": best_alpha,
            "best_alpha_idm": 1.0 - best_alpha,
            "DATA_DIR": str(DATA_DIR),
            "OUTPUT_DIR": str(OUTPUT_DIR),
            "RUN_DIR": str(run_dir),
            "best_global_params": best_global_params,
            "best_params_braking": best_params_braking,
            "best_params_accel": best_params_accel,
            "best_params_steady": best_params_steady,
        }, f, indent=2)

    print("\n=== WEIGHTED FUSION SUMMARY ===")
    print(fusion_summary_df.to_string(index=False))

    print("\n=== WEIGHTED FUSION REGIME METRICS ===")
    print(fusion_regime_metrics_df.to_string(index=False))

    print("\nSaved:")
    print(summary_path)
    print(split_path)
    print(regime_path)
    print(predictions_path)
    print(alpha_path)
    print(config_path)

    suite_rows.append(fusion_summary_df.iloc[0].to_dict())

    temp_regime = fusion_regime_metrics_df.copy()
    temp_regime["ExperimentName"] = experiment_name
    temp_regime["PriorMode"] = prior_mode
    temp_regime["Model"] = model_label
    suite_regime_rows.append(temp_regime)

    suite_alpha_rows.append(alpha_search_df)

    # Cleanup memory before next prior experiment
    del meta_train_p, meta_val_p, meta_test_p
    del idm_train, idm_val, idm_test
    del fusion_train_pred, fusion_val_pred, fusion_test_pred
    del fusion_test_results, fusion_summary_df, fusion_split_metrics_df
    del fusion_regime_metrics_df, alpha_search_df
    gc.collect()

# ============================================================
# SAVE SUITE-LEVEL SUMMARY
# ============================================================

weighted_fusion_suite_summary_df = pd.DataFrame(suite_rows)
weighted_fusion_suite_summary_df = weighted_fusion_suite_summary_df.sort_values("MAE").reset_index(drop=True)

weighted_fusion_suite_regime_df = pd.concat(suite_regime_rows, ignore_index=True)
weighted_fusion_suite_alpha_df = pd.concat(suite_alpha_rows, ignore_index=True)

suite_summary_path = OUTPUT_DIR / f"weighted_fusion_suite_summary_{SEQUENCE_TAG}_{THRESHOLD_TAG}_{IDM_PARAM_TAG}.csv"
suite_regime_path = OUTPUT_DIR / f"weighted_fusion_suite_regime_metrics_{SEQUENCE_TAG}_{THRESHOLD_TAG}_{IDM_PARAM_TAG}.csv"
suite_alpha_path = OUTPUT_DIR / f"weighted_fusion_suite_alpha_search_{SEQUENCE_TAG}_{THRESHOLD_TAG}_{IDM_PARAM_TAG}.csv"

weighted_fusion_suite_summary_df.to_csv(suite_summary_path, index=False)
weighted_fusion_suite_regime_df.to_csv(suite_regime_path, index=False)
weighted_fusion_suite_alpha_df.to_csv(suite_alpha_path, index=False)

print("\n" + "=" * 80)
print("FINAL WEIGHTED FUSION SUITE SUMMARY")
print("=" * 80)
print(
    weighted_fusion_suite_summary_df[
        [
            "ExperimentName",
            "Model",
            "PriorMode",
            "alpha_transformer",
            "alpha_idm",
            "MAE",
            "RMSE",
            "MedAE",
            "R2",
            "Bias",
            "Corr"
        ]
    ].to_string(index=False)
)

print("\nSaved suite outputs:")
print(suite_summary_path)
print(suite_regime_path)
print(suite_alpha_path)


Experiment: TransformerWeightedFusion_GlobalIDM_H25_Hz75_S1_brake0p1_accel0p2_refinedIDM_v2_currentStatePrior_v2
Prior mode: global
Run dir: C:\Users\ANARAMO\Documents\Tese\Tese\Tese\HybridMobility\processed_data_final\transformer_weighted_fusion_runs\TransformerWeightedFusion_GlobalIDM_H25_Hz75_S1_brake0p1_accel0p2_refinedIDM_v2_currentStatePrior_v2
Prior metadata aligned correctly with Transformer targets.
Best alpha selected on validation: 1.0
Transformer weight: 1.0
IDM weight: 0.0

=== WEIGHTED FUSION SUMMARY ===
                                                                                      ExperimentName                                     Model PriorMode  BrakeThreshold  AccelThreshold  alpha_transformer  alpha_idm      MAE    RMSE    MedAE       R2     Bias     Corr
TransformerWeightedFusion_GlobalIDM_H25_Hz75_S1_brake0p1_accel0p2_refinedIDM_v2_currentStatePrior_v2 Weighted Fusion: Transformer + Global IDM    global            -0.1             0.2                1.0    